# Qwen 2.5 Intent Distillation Pipeline
**Google Colab T4 GPU — Bulletproof Edition**

Pipeline:
1. Dataset Preparation
2. Teacher Pseudo-Labeling (Qwen2.5-7B-Instruct, 4-bit)
3. Student QLoRA Fine-Tuning (Qwen2.5-0.5B-Instruct)
4. Evaluation (Accuracy / Macro-F1)
5. CPU Edge Benchmark (Latency / RAM)

In [ ]:
!pip install -q transformers==4.46.3 datasets accelerate peft==0.13.2 trl==0.12.2 bitsandbytes scikit-learn psutil tqdm

## 1. Dataset Preparation

In [ ]:
import json, random, os

os.makedirs('data/qwen_distill', exist_ok=True)
os.makedirs('reports', exist_ok=True)

gold_samples, unlabeled_samples = [], []
with open('data/raw/dataset_final.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        if not line.strip(): continue
        d = json.loads(line)
        if d.get('type') == 'unlabeled':
            unlabeled_samples.append({'text': d['text']})
        else:
            gold_samples.append({'text': d['text'], 'label': d['intent']})

hard_test_samples = []
with open('data/raw/dataset_150_bonus_v2.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        if not line.strip(): continue
        d = json.loads(line)
        hard_test_samples.append({'text': d['text'], 'label': d['intent']})

random.seed(42)
random.shuffle(gold_samples)
normal_test_samples = gold_samples[:50]
train_gold_samples  = gold_samples[50:]

def save_jsonl(data, path):
    with open(path, 'w', encoding='utf-8') as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')

save_jsonl(train_gold_samples,  'data/qwen_distill/train_gold.jsonl')
save_jsonl(unlabeled_samples,   'data/qwen_distill/unlabeled.jsonl')
save_jsonl(normal_test_samples, 'data/qwen_distill/normal_test.jsonl')
save_jsonl(hard_test_samples,   'data/qwen_distill/hard_test.jsonl')

print(f'Train Gold:  {len(train_gold_samples)}')
print(f'Unlabeled:   {len(unlabeled_samples)}')
print(f'Normal Test: {len(normal_test_samples)}')
print(f'Hard Test:   {len(hard_test_samples)}')

## 2. System Prompt

In [ ]:
SYSTEM_PROMPT = """You are an expert intent classifier for a Vietnamese vending machine assistant.
Analyze the user's input text (which may contain ASR noise) and output ONLY the intent name.

Valid Intents:
- greeting
- show_menu
- buy_product
- add_product
- change_product
- payment
- cancel
- help
- unknown

Example:
Text: "cho tôi một lon coca"
Intent: buy_product
"""

INTENTS = ['greeting','show_menu','buy_product','add_product',
           'change_product','payment','cancel','help','unknown']

def parse_intent(response_text):
    response_text = response_text.strip().lower()
    for intent in INTENTS:
        if intent in response_text:
            return intent
    return 'unknown'

## 3. Teacher Pseudo-Labeling (Qwen2.5-7B-Instruct)

In [ ]:
import torch, gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from tqdm.notebook import tqdm

torch.cuda.empty_cache(); gc.collect()

TEACHER = 'Qwen/Qwen2.5-7B-Instruct'

bnb4 = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
)

t_tok = AutoTokenizer.from_pretrained(TEACHER)
t_model = AutoModelForCausalLM.from_pretrained(
    TEACHER, quantization_config=bnb4,
    device_map='auto', torch_dtype=torch.float16,
)

pseudo_labeled = []
for item in tqdm(unlabeled_samples, desc='Teacher labeling'):
    msgs = [
        {'role':'system','content': SYSTEM_PROMPT},
        {'role':'user',  'content': f'Text: "{item["text"]}"\nIntent:'},
    ]
    inp = t_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = t_tok([inp], return_tensors='pt').to(t_model.device)
    out = t_model.generate(**ids, max_new_tokens=10, do_sample=False)
    new_tokens = out[0][ids.input_ids.shape[1]:]
    resp = t_tok.decode(new_tokens, skip_special_tokens=True)
    pseudo_labeled.append({'text': item['text'], 'label': parse_intent(resp)})

save_jsonl(pseudo_labeled, 'data/qwen_distill/pseudo_labeled.jsonl')

train_distill = train_gold_samples + pseudo_labeled
save_jsonl(train_distill, 'data/qwen_distill/train_distill.jsonl')
print(f'train_distill.jsonl: {len(train_distill)} samples')

del t_model, t_tok; torch.cuda.empty_cache(); gc.collect()
print('Teacher model released.')

## 4. Student Fine-Tuning (QLoRA)
**Key fix:** `fp16=False, bf16=False` — disables AMP entirely.
The 0.5B model in 4-bit + LoRA uses < 2 GB VRAM, so fp32 training fits easily on T4.

In [ ]:
import torch, gc
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

torch.cuda.empty_cache(); gc.collect()

STUDENT = 'Qwen/Qwen2.5-0.5B-Instruct'

# ── Tokenizer ──
s_tok = AutoTokenizer.from_pretrained(STUDENT)
s_tok.pad_token = s_tok.eos_token

# ── Format dataset ──
rows = []
for item in train_distill:
    msgs = [
        {'role':'system',   'content': SYSTEM_PROMPT},
        {'role':'user',     'content': f'Text: "{item["text"]}"\nIntent:'},
        {'role':'assistant','content': item['label']},
    ]
    rows.append({'text': s_tok.apply_chat_template(msgs, tokenize=False)})
train_dataset = Dataset.from_list(rows)

# ── Load base model (4-bit) ──
bnb4 = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
s_model = AutoModelForCausalLM.from_pretrained(
    STUDENT, quantization_config=bnb4,
    device_map='auto', torch_dtype=torch.float16,
)

# ── Prepare for k-bit training ──
s_model = prepare_model_for_kbit_training(s_model)

# ── Apply LoRA ──
lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=['q_proj','k_proj','v_proj','o_proj'],
    bias='none', task_type='CAUSAL_LM',
)
s_model = get_peft_model(s_model, lora_cfg)

# ════════════════════════════════════════════════════════
# NUCLEAR FIX: cast every single tensor in the model
# (parameters + buffers + tied weights) away from bfloat16
# ════════════════════════════════════════════════════════
for name, param in s_model.named_parameters():
    if param.data.dtype == torch.bfloat16:
        param.data = param.data.to(torch.float16)
for name, buf in s_model.named_buffers():
    if buf.dtype == torch.bfloat16:
        buf.data = buf.data.to(torch.float16)

s_model.print_trainable_parameters()

# ── Training config ──
# *** fp16=False AND bf16=False ***
# This disables PyTorch AMP completely, so the BFloat16
# GradScaler crash CANNOT happen. The 0.5B QLoRA model
# is tiny enough to train in fp32 on a T4 with no OOM.
training_args = SFTConfig(
    dataset_text_field='text',
    output_dir='./student_qwen',
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=10,
    save_strategy='epoch',
    optim='paged_adamw_32bit',
    fp16=False,   # ← AMP OFF
    bf16=False,   # ← AMP OFF
    report_to='none',
)

trainer = SFTTrainer(
    model=s_model,
    train_dataset=train_dataset,
    processing_class=s_tok,
    args=training_args,
    # peft_config is NOT passed here because model is already wrapped
)

print('Starting student fine-tuning (fp32, no AMP)...')
trainer.train()

trainer.model.save_pretrained('./student_qwen_adapter')
s_tok.save_pretrained('./student_qwen_adapter')
print('Adapter saved to ./student_qwen_adapter')

del trainer, s_model; torch.cuda.empty_cache(); gc.collect()

## 5. Evaluation

In [ ]:
import torch, gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from tqdm.notebook import tqdm

STUDENT = 'Qwen/Qwen2.5-0.5B-Instruct'
ADAPTER = './student_qwen_adapter'

def evaluate(model, tokenizer, data, label):
    y_true, y_pred = [], []
    for item in tqdm(data, desc=label):
        msgs = [
            {'role':'system','content': SYSTEM_PROMPT},
            {'role':'user',  'content': f'Text: "{item["text"]}"\nIntent:'},
        ]
        inp = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        ids = tokenizer([inp], return_tensors='pt').to(model.device)
        out = model.generate(**ids, max_new_tokens=10, do_sample=False)
        new_tokens = out[0][ids.input_ids.shape[1]:]
        resp = tokenizer.decode(new_tokens, skip_special_tokens=True)
        y_true.append(item['label'])
        y_pred.append(parse_intent(resp))
    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, average='macro', zero_division=0)
    print(f'\n{label}  Acc={acc:.4f}  F1={f1:.4f}')
    return acc, f1

bnb4 = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
)
base = AutoModelForCausalLM.from_pretrained(
    STUDENT, quantization_config=bnb4,
    device_map='auto', torch_dtype=torch.float16,
)
eval_model = PeftModel.from_pretrained(base, ADAPTER)
eval_tok   = AutoTokenizer.from_pretrained(ADAPTER)

evaluate(eval_model, eval_tok, normal_test_samples, 'Normal Test')
evaluate(eval_model, eval_tok, hard_test_samples,   'Hard Test')

del eval_model, base; torch.cuda.empty_cache(); gc.collect()

## 6. CPU Edge Benchmark

In [ ]:
import time, psutil, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

STUDENT = 'Qwen/Qwen2.5-0.5B-Instruct'
ADAPTER = './student_qwen_adapter'

def latency(model, tokenizer, text):
    msgs = [
        {'role':'system','content': SYSTEM_PROMPT},
        {'role':'user',  'content': f'Text: "{text}"\nIntent:'},
    ]
    inp = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = tokenizer([inp], return_tensors='pt')
    t0 = time.time()
    _ = model.generate(**ids, max_new_tokens=10, do_sample=False)
    return time.time() - t0

t0 = time.time()
cpu_base  = AutoModelForCausalLM.from_pretrained(STUDENT, device_map='cpu', torch_dtype=torch.float32)
cpu_model = PeftModel.from_pretrained(cpu_base, ADAPTER)
cpu_tok   = AutoTokenizer.from_pretrained(ADAPTER)
load_s = time.time() - t0

mem_mb = psutil.Process().memory_info().rss / 1024**2
print(f'Load: {load_s:.1f}s | RAM: {mem_mb:.0f} MB')

latency(cpu_model, cpu_tok, 'warmup')  # warmup
tests = ['mua 1 chai nước suối','thôi không mua nữa','có nước gì ngon không','thanh toán momo']
lats = [latency(cpu_model, cpu_tok, t) for t in tests]
print(f'Avg CPU latency: {sum(lats)/len(lats):.3f} s/query')